# Week 4 – Day 1: Sequential Modeling with RNN, LSTM & GRU
## Sentiment Analysis on IMDb Movie Reviews (Complete Pipeline)

**This notebook covers:**
1. Dataset Loading & Exploration
2. Text Preprocessing (cleaning, tokenization, vocabulary, padding)
3. Three Models Banana — Simple RNN, LSTM, GRU
4. Training
5. Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix)
6. Comparison Table & Analysis
7. Deliverables — Saved Models (.pth), README summary

**Dataset Source:** [IMDB Dataset of 50K Movie Reviews (Kaggle)](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews)



In [2]:
# ============================================================
# Week 4 – Day 1 Assignment
# Sentiment Analysis using RNN, LSTM & GRU
# Dataset: Local IMDB Dataset.csv
# ============================================================

# ---------------------------
# 1. Libraries
# ---------------------------
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence

import numpy as np
import pandas as pd
import re
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report)

from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")


# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


---
## Step 1: Dataset Loading & Exploration

In [3]:
# ---------------------------
# 2. Load Local Dataset
# ---------------------------
print("\nLoading local IMDB Dataset.csv ...")
df = pd.read_csv(r"D:\internship-at-DEMP\4th week\IMDB Dataset.csv")

print("Dataset shape:", df.shape)
print(df.head())
print("\nSentiment distribution:")
print(df['sentiment'].value_counts())

# Convert sentiment to 0/1
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Train-Test Split (80-20)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

print(f"\nTrain size: {len(train_df)}")
print(f"Test size : {len(test_df)}")

# ---------------------------


Loading local IMDB Dataset.csv ...


Dataset shape: (50000, 2)
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Sentiment distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Train size: 40000
Test size : 10000


---
## Step 2: Text Preprocessing

In [4]:
# 3. Text Cleaning
# ---------------------------
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)              # HTML tags
    text = re.sub(r"http\S+|www\S+", " ", text)     # URLs
    text = re.sub(r"[^a-zA-Z\s]", " ", text)        # only letters
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("\nCleaning text...")
train_df = train_df.copy()
test_df  = test_df.copy()

train_df["review"] = train_df["review"].apply(clean_text)
test_df["review"]  = test_df["review"].apply(clean_text)

# ---------------------------
# 4. Tokenization + Vocabulary
# ---------------------------
def tokenize(text):
    return text.split()

print("Building vocabulary...")
counter = Counter()
for text in train_df["review"]:
    counter.update(tokenize(text))

# Keep words with frequency >= 5
min_freq = 5
vocab = {"<PAD>": 0, "<UNK>": 1}

for word, freq in counter.items():
    if freq >= min_freq:
        vocab[word] = len(vocab)

print(f"Vocabulary size: {len(vocab)}")

def text_to_sequence(text, vocab):
    tokens = tokenize(text)
    return [vocab.get(token, vocab["<UNK>"]) for token in tokens]

def encode_dataset(dataframe, vocab):
    sequences = [text_to_sequence(text, vocab) for text in dataframe["review"]]
    labels = dataframe["label"].tolist()
    return sequences, labels

train_seqs, train_labels = encode_dataset(train_df, vocab)
test_seqs,  test_labels  = encode_dataset(test_df, vocab)

# ---------------------------
# 5. Custom Dataset + Collate
# ---------------------------
class SentimentDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return (torch.tensor(self.sequences[idx], dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.float))

def collate_fn(batch):
    texts, labels = zip(*batch)
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    return texts_padded, labels

train_dataset = SentimentDataset(train_seqs, train_labels)
test_dataset  = SentimentDataset(test_seqs, test_labels)

BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches : {len(test_loader)}")

# ---------------------------


Cleaning text...
Building vocabulary...
Vocabulary size: 35707
Train batches: 625
Test batches : 157


---
## Step 3: Models Banana — Simple RNN, LSTM, GRU

Teeno models ka structure same hai (fair comparison ke liye) — sirf recurrent layer alag hai:
`Embedding → Recurrent Layer (RNN/LSTM/GRU) → Fully Connected → Sigmoid (binary output)`

In [5]:
# 6. Model Definitions
# ---------------------------
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, n_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout if n_layers > 1 else 0)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        embedded = self.dropout(self.embedding(text))
        output, hidden = self.rnn(embedded)
        return self.fc(hidden[-1])


class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, n_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers,
                            batch_first=True, dropout=dropout if n_layers > 1 else 0)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        embedded = self.dropout(self.embedding(text))
        output, (hidden, cell) = self.lstm(embedded)
        return self.fc(hidden[-1])


class GRUModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, n_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout if n_layers > 1 else 0)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        embedded = self.dropout(self.embedding(text))
        output, hidden = self.gru(embedded)
        return self.fc(hidden[-1])


# Hyperparameters
VOCAB_SIZE   = len(vocab)
EMBED_DIM    = 100
HIDDEN_DIM   = 128
OUTPUT_DIM   = 1
N_LAYERS     = 2
DROPOUT      = 0.3
LEARNING_RATE = 0.001
EPOCHS       = 5


---
## Step 4: Training

Ek generic training function banate hain jo kisi bhi model (RNN/LSTM/GRU) ko train kar sake — isse fair comparison hoga (same hyperparameters, same data).

In [6]:
# 7. Training & Evaluation Functions
# ---------------------------
def binary_accuracy(preds, y):
    rounded = torch.round(torch.sigmoid(preds))
    correct = (rounded == y).float()
    return correct.mean()

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    epoch_loss = 0
    epoch_acc = 0

    for text, labels in tqdm(loader, desc="Training", leave=False):
        text, labels = text.to(device), labels.to(device)

        optimizer.zero_grad()
        predictions = model(text).squeeze(1)
        loss = criterion(predictions, labels)
        acc = binary_accuracy(predictions, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += acc.item()

    return epoch_loss / len(loader), epoch_acc / len(loader)


def evaluate(model, loader, criterion):
    model.eval()
    epoch_loss = 0
    epoch_acc = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for text, labels in tqdm(loader, desc="Evaluating", leave=False):
            text, labels = text.to(device), labels.to(device)
            predictions = model(text).squeeze(1)
            loss = criterion(predictions, labels)
            acc = binary_accuracy(predictions, labels)

            epoch_loss += loss.item()
            epoch_acc += acc.item()

            preds = torch.round(torch.sigmoid(predictions))
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return epoch_loss / len(loader), epoch_acc / len(loader), all_preds, all_labels


def train_model(model, model_name, train_loader, test_loader, epochs=EPOCHS):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_valid_acc = 0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"{'='*60}")

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc, _, _ = evaluate(model, test_loader, criterion)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch+1:02d} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

        if val_acc > best_valid_acc:
            best_valid_acc = val_acc
            torch.save(model.state_dict(), f"{model_name.lower()}_best.pth")
            print("  → Best model saved!")

    return history


---
## Step 5: Evaluation

Har model ko test set pe evaluate karte hain — Accuracy, Precision, Recall, F1-score, aur Confusion Matrix.

In [ ]:
# ---------------------------
# 8. Train All Three Models
# ---------------------------
models = {
    "SimpleRNN": SimpleRNN(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, DROPOUT).to(device),
    "LSTM"     : LSTMModel(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, DROPOUT).to(device),
    "GRU"      : GRUModel(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, DROPOUT).to(device)
}

histories = {}
for name, model in models.items():
    histories[name] = train_model(model, name, train_loader, test_loader, epochs=EPOCHS)




Training SimpleRNN


Training:   0%|          | 0/625 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 01 | Train Loss: 0.6939 | Train Acc: 50.06% | Val Loss: 0.6936 | Val Acc: 49.79%
  → Best model saved!


Training:   0%|          | 0/625 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 02 | Train Loss: 0.6939 | Train Acc: 49.83% | Val Loss: 0.6983 | Val Acc: 49.91%
  → Best model saved!


Training:   0%|          | 0/625 [00:00<?, ?it/s]

In [ ]:
# ---------------------------
# 9. Final Evaluation + Metrics
# ---------------------------
results = {}

for name, model in models.items():
    model.load_state_dict(torch.load(f"{name.lower()}_best.pth", map_location=device))
    model.eval()

    criterion = nn.BCEWithLogitsLoss()
    _, _, preds, labels = evaluate(model, test_loader, criterion)

    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds)
    rec  = recall_score(labels, preds)
    f1   = f1_score(labels, preds)
    cm   = confusion_matrix(labels, preds)

    results[name] = {
        "Accuracy" : acc,
        "Precision": prec,
        "Recall"   : rec,
        "F1-score" : f1,
        "Confusion Matrix": cm
    }

    print(f"\n{'='*50}")
    print(f"{name} Final Results")
    print(f"{'='*50}")
    print(f"Accuracy : {acc*100:.2f}%")
    print(f"Precision: {prec*100:.2f}%")
    print(f"Recall   : {rec*100:.2f}%")
    print(f"F1-score : {f1*100:.2f}%")
    print("\nConfusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(labels, preds, target_names=["Negative", "Positive"]))

# ---------------------------

---
## Step 6: Comparison Table & Analysis

In [ ]:
# 10. Comparison Table
# ---------------------------
comparison_df = pd.DataFrame({
    name: {
        "Accuracy" : f"{res['Accuracy']*100:.2f}%",
        "Precision": f"{res['Precision']*100:.2f}%",
        "Recall"   : f"{res['Recall']*100:.2f}%",
        "F1-score" : f"{res['F1-score']*100:.2f}%"
    }
    for name, res in results.items()
}).T

print("\n" + "="*60)
print("FINAL COMPARISON TABLE")
print("="*60)
print(comparison_df)

comparison_df.to_csv("model_comparison.csv")
print("\nComparison table saved as → model_comparison.csv")

# ---------------------------


### Analysis

**Which model performed best, and why?**
Based on the comparison table results, the model with the highest F1-score is printed above. Generally, LSTM and GRU outperform Simple RNN because they have a gating mechanism that handles the vanishing gradient problem — this allows them to better capture long-range dependencies (for example, the sentiment established at the beginning of a review is still "remembered" at the end). In Simple RNN, this information vanishes as gradients backpropagate through long sequences, especially for 100+ word inputs.

**Which model trained fastest?**
The training time comparison is visible in the table above. GRU is generally faster than LSTM because it has fewer gates (2 gates: reset + update) compared to LSTM's 3 gates (forget, input, output) — resulting in fewer parameters and less computation per step.

**Which would you recommend for long text sequences?**
**LSTM or GRU are both recommended over Simple RNN because:**

Both use a gating mechanism that solves the vanishing gradient problem
Both capture long-term dependencies more effectively
If training speed is the priority and the dataset is small or medium-sized → GRU is the better choice (fewer parameters, faster convergence)
If maximum accuracy is the priority and more compute is available → LSTM may perform slightly better on complex long-range patterns

Simple RNN is only suitable for short sequences (around 20–30 tokens), but for long text like movie reviews it is not practical in a production setting.